# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeddaniyalg/flyrank-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit of Analysis (Grain):** One row represents one `content_hash_id` for a specific `client_hash_id` on a specific `report_date`.
* **Time Window:** For development, we will use a mid-panel month: **March 2026** (`2026-03`). The final month of the dataset is reserved strictly as a sealed test set.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
fact_table = f"{rel}/fact_content_daily_performance/**/*.parquet"

grain_check = con.sql(f"""
    WITH march_sample AS (
        SELECT client_hash_id, content_hash_id, report_date
        FROM read_parquet('{fact_table}')
        WHERE report_date >= DATE '2026-03-01' 
          AND report_date < DATE '2026-04-01'
        LIMIT 50000
    )
    SELECT 
        COUNT(*) AS total_sample_rows,
        COUNT(DISTINCT client_hash_id || '_' || content_hash_id || '_' || CAST(report_date AS VARCHAR)) AS unique_grain_rows
    FROM march_sample
""").df()

print("1. Grain Verification:")
print(grain_check)

1. Grain Verification:
   total_sample_rows  unique_grain_rows
0              50000              50000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Features (Inputs):** `gsc_impressions`, `gsc_avg_position`, `gsc_sum_position`, `is_indexed`, `has_traffic`. (These metrics represent historical search engine visibility prior to our target action).
* **Label / Target:** `gsc_clicks` (This is the future traffic event we are attempting to model/predict).
* **Context / Meta:** `report_date`, `client_hash_id`, `content_hash_id` (Needed for joining and grouping, but not mathematical features).
* **Excluded:** `ga4_sessions` and `ga4_pageviews` from the target prediction window. **Why:** Using engagement metrics from the same time period we are predicting clicks causes target leakage, allowing the model to "cheat".

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

availability_check = con.sql(f"""
    SELECT 
        COUNT(*) AS total_slice_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(CASE WHEN (gsc_impressions > 0 IS TRUE) THEN 1 END) AS rows_with_impressions,
        COUNT(CASE WHEN (gsc_data_available IS TRUE) THEN 1 END) AS rows_with_gsc_data
    FROM read_parquet('{fact_table}')
    WHERE report_date >= DATE '2026-03-01' 
      AND report_date < DATE '2026-04-01'
""").df()

print("2. Slice Verification & Availability (IS TRUE):")
print(availability_check)

2. Slice Verification & Availability (IS TRUE):
   total_slice_rows   min_date   max_date  rows_with_impressions  \
0           9841378 2026-03-01 2026-03-31                3611061   

   rows_with_gsc_data  
0             3611061  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**The Five-Feature Frame:**
1. `gsc_impressions`: Knowable at the decision moment because historical search exposure is logged.
2. `gsc_avg_position`: Knowable at the decision moment because the content's historical ranking is observed.
3. `gsc_sum_position`: Knowable at the decision moment as an aggregate historical ranking metric.
4. `is_indexed`: Knowable at the decision moment because a non-zero position indicates the URL was crawled.
5. `has_traffic`: Knowable at the decision moment because past non-zero clicks are a matter of record.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

df_march = con.sql(f"""
    SELECT 
        report_date,
        gsc_impressions,
        gsc_avg_position,
        gsc_sum_position,
        gsc_clicks,
        ga4_sessions
    FROM read_parquet('{fact_table}')
    WHERE report_date >= DATE '2026-03-01' 
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE
    LIMIT 20000
""").df()

df_features = pd.DataFrame({
    'gsc_impressions': df_march['gsc_impressions'],
    'gsc_avg_position': df_march['gsc_avg_position'],
    'gsc_sum_position': df_march['gsc_sum_position'],
    'is_indexed': (df_march['gsc_avg_position'] > 0).astype(int),
    'has_traffic': (df_march['gsc_clicks'] > 0).astype(int)
})
y = df_march['gsc_clicks'] 

df_features_leaked = df_features.copy()
df_features_leaked['TRAP_same_day_sessions'] = df_march['ga4_sessions']

honest_corr = df_features['gsc_impressions'].corr(y)
leaked_corr = df_features_leaked['TRAP_same_day_sessions'].corr(y)

print("--- 3. Leakage Trap Experiment ---")
print(f"Honest Correlation (impressions vs clicks): {honest_corr:.4f}")
print(f"Leaked Score (TRAP_same_day_sessions vs clicks): {leaked_corr:.4f}")

del df_features_leaked['TRAP_same_day_sessions']
print("Trap removed. Honest feature frame preserved.")

--- 3. Leakage Trap Experiment ---
Honest Correlation (impressions vs clicks): 0.6896
Leaked Score (TRAP_same_day_sessions vs clicks): 0.8757
Trap removed. Honest feature frame preserved.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Data Limitations:** This slice represents a daily aggregation of Search Console data. It cannot tell us intraday ranking volatility (e.g., if a page ranked #1 for two hours and then dropped to #50). It also cannot identify external algorithmic updates by Google or offline SEO factors (like backlink acquisition) that are not native to Search Console. Furthermore, the panel is unbalanced; clients have different `gsc_data_start` times.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
limitations_check = con.sql(f"""
    SELECT 
        client_hash_id,
        COUNT(DISTINCT report_date) AS active_days,
        COUNT(*) AS total_entries
    FROM read_parquet('{fact_table}')
    WHERE report_date >= DATE '2026-03-01' 
      AND report_date < DATE '2026-04-01'
    GROUP BY client_hash_id
    ORDER BY active_days ASC
    LIMIT 5
""").df()

print("4. Limitations Check (Unbalanced observation days across clients):")
print(limitations_check)

4. Limitations Check (Unbalanced observation days across clients):
            client_hash_id  active_days  total_entries
0  client_e00b29e582949543            9           1216
1  client_810019792c9b8efc           12           1812
2  client_f6f0cdf26d03d7bd           13            520
3  client_86ebc2f12c01f586           29           8823
4  client_0fa64a184f18a4a0           31          44197


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.